In [1]:
!pip install -q -U transformers peft bitsandbytes accelerate trl datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.6/10.6 MB 97.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 680.7/680.7 kB 41.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 32.4 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 383.7/383.7 kB 27.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 751.0/751.0 kB 49.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.0/529.0 kB 34.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 661.5/661.5 kB 45.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 117.5 MB/s eta 0:00:0000:01


In [2]:
%%writefile train.py
import os
import torch
from datasets import load_dataset, concatenate_datasets
from transformers import (
    AutoTokenizer, 
    AutoModelForCausalLM, 
    BitsAndBytesConfig,
    TrainingArguments,
    Trainer,                            # <-- Sử dụng Trainer lõi
    DataCollatorForLanguageModeling     # <-- Công cụ gom batch và tạo label tự động
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from accelerate import Accelerator

os.environ["TOKENIZERS_PARALLELISM"] = "false"

def prepare_stratified_datasets(file_path):
    dataset = load_dataset("json", data_files=file_path)["train"]
    unique_sources = list(set(dataset["source"]))
    
    train_lists, val_lists, test_lists = [], [], []
    for src in unique_sources:
        sub_ds = dataset.filter(lambda x: x["source"] == src)
        if len(sub_ds) < 3:
            train_lists.append(sub_ds)
            continue
            
        split_1 = sub_ds.train_test_split(test_size=0.2, seed=42)
        train_lists.append(split_1["train"])
        
        split_2 = split_1["test"].train_test_split(test_size=0.5, seed=42)
        val_lists.append(split_2["train"])
        test_lists.append(split_2["test"])

    train_dataset = concatenate_datasets(train_lists).shuffle(seed=42)
    val_dataset = concatenate_datasets(val_lists)
    test_dataset = concatenate_datasets(test_lists)
    return train_dataset, val_dataset, test_dataset

def main():
    accelerator = Accelerator()
    
    if accelerator.is_main_process:
        print("Đang tải và chia dữ liệu...")
        
    data_path = "/kaggle/input/datasets/truongminh3105/nwp-dataset/final_rich_dataset.jsonl"
    train_ds, val_ds, test_ds = prepare_stratified_datasets(data_path)

    model_id = "Qwen/Qwen2-1.5B"
    
    tokenizer = AutoTokenizer.from_pretrained(model_id, padding_side="right")
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    # 1. Tự động Tokenize dữ liệu trước khi train
    def tokenize_func(examples):
        return tokenizer(
            examples["segmented_text"], 
            truncation=True, 
            max_length=256
        )
    
    if accelerator.is_main_process:
        print("Đang Tokenize dữ liệu...")
        
    # main_process_first giúp tránh đụng độ bộ nhớ khi chạy song song 2 GPU
    with accelerator.main_process_first():
        train_tokenized = train_ds.map(tokenize_func, batched=True, remove_columns=train_ds.column_names)
        val_tokenized = val_ds.map(tokenize_func, batched=True, remove_columns=val_ds.column_names)

    # 2. Cấu hình QLoRA
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
    )

    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        quantization_config=bnb_config,
        device_map={"": accelerator.local_process_index},
        trust_remote_code=True
    )
    
    model.config.use_cache = False
    model.gradient_checkpointing_enable()
    
    # 3. Chuẩn bị Model cho 4-bit và nhúng Adapter LoRA (thay thế chức năng SFTTrainer)
    model = prepare_model_for_kbit_training(model)
    peft_config = LoraConfig(
        r=16,
        lora_alpha=32,
        target_modules=["q_proj", "v_proj"],
        bias="none",
        task_type="CAUSAL_LM"
    )
    model = get_peft_model(model, peft_config)

    # 4. Tham số huấn luyện
    training_args = TrainingArguments(
        output_dir="/kaggle/working/qwen_nwp_checkpoints",
        per_device_train_batch_size=4,        
        gradient_accumulation_steps=4,        
        learning_rate=2e-4,
        fp16=True,                            
        logging_steps=10,
        
        # SỬA Ở ĐÂY: Dừng sau 2000 bước (Thay vì -1)
        max_steps=2000,                         
        
        num_train_epochs=1,                   
        optim="paged_adamw_32bit",            
        eval_strategy="steps",                
        eval_steps=100,      # Tăng lên 100 để đỡ mất thời gian eval liên tục
        save_strategy="steps",
        save_steps=500,      # Lưu checkpoint mỗi 500 bước cho an toàn
        ddp_find_unused_parameters=False,     
        report_to="none"
    )

    # DataCollator tự động tạo mask và dịch nhãn (shift labels) cho Next Word Prediction
    data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

    # 5. Dùng Trainer gốc (Cực kỳ ổn định)
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_tokenized,
        eval_dataset=val_tokenized,
        data_collator=data_collator,
    )

    trainer.train()

    # Chỉ lưu model ở tiến trình chính
    if accelerator.is_main_process:
        trainer.model.save_pretrained("/kaggle/working/qwen_nwp_final")
        tokenizer.save_pretrained("/kaggle/working/qwen_nwp_final")
        print("Đã lưu model LoRA thành công!")

if __name__ == "__main__":
    main()

Writing train.py


In [ ]:
!accelerate launch --multi_gpu --num_processes=2 train.py

The following values were not passed to `accelerate launch` and had defaults used instead:
	`--num_machines` was set to a value of `1`
	`--mixed_precision` was set to a value of `'no'`
	`--dynamo_backend` was set to a value of `'no'`
To avoid this warning pass in values for each of the problematic parameters or run `accelerate config`.
Đang tải và chia dữ liệu...
Generating train split: 797461 examples [00:07, 113256.16 examples/s]
config.json: 100%|█████████████████████████████| 662/662 [00:00<00:00, 3.26MB/s]
tokenizer_config.json: 1.29kB [00:00, 3.79MB/s]0:01<00:04, 133465.43 examples/s]
vocab.json: 2.78MB [00:00, 79.3MB/s]0/797461 [00:01<00:04, 126719.29 examples/s]
merges.txt: 1.67MB [00:00, 78.4MB/s]0/797461 [00:01<00:04, 114095.28 examples/s]
tokenizer.json: 7.03MB [00:00, 111MB/s]97461 [00:02<00:05, 102438.37 examples/s]
Map: 100%|███████████████████████| 79745/79745 [00:16<00:00, 4827.10 examples/s]
model.safetensors: 100%|████████████████████| 3.09G/3.09G [00:12<00:00, 251MB/

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel

def load_inference_model(base_model_id, lora_path):
    print("Đang load model phục vụ Inference...")
    
    # Load Tokenizer
    tokenizer = AutoTokenizer.from_pretrained(lora_path)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    
    # Cấu hình 4-bit để tiết kiệm VRAM (Quan trọng nếu GPU đang bị chiếm bởi bản train)
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True
    )
    
    # Load Base Model
    base_model = AutoModelForCausalLM.from_pretrained(
        base_model_id,
        quantization_config=bnb_config,
        device_map="auto", 
        trust_remote_code=True
    )
    
    # Kết hợp Adapter LoRA
    model = PeftModel.from_pretrained(base_model, lora_path)
    model.eval()
    return tokenizer, model

# Khởi tạo (Chỉ chạy 1 lần)
model_path = "/kaggle/working/qwen_nwp_final"
tokenizer_inf, model_inf = load_inference_model("Qwen/Qwen2-1.5B", model_path)

def suggest_next_words(prompt_text, num_words=10):
    # Tiền xử lý: Chuyển prompt sang dạng segmented nếu bạn train bằng segmented_text
    # prompt_segmented = prompt_text.replace(" ", "_") 
    
    inputs = tokenizer_inf(prompt_text, return_tensors="pt").to(model_inf.device)
    
    with torch.no_grad():
        outputs = model_inf.generate(
            **inputs,
            max_new_tokens=num_words,
            num_beams=5,
            no_repeat_ngram_size=2,
            early_stopping=True,
            pad_token_id=tokenizer_inf.pad_token_id,
            eos_token_id=tokenizer_inf.eos_token_id,
            # Thêm do_sample=True nếu muốn kết quả đa dạng hơn, 
            # nhưng Beam Search (num_beams > 1) thường dùng kết quả deterministic
            do_sample=False 
        )
        
    full_text = tokenizer_inf.decode(outputs[0], skip_special_tokens=True)
    suggested_part = full_text[len(prompt_text):].strip()
    
    # Hậu xử lý: Trả về tiếng Việt tự nhiên (bỏ dấu gạch dưới)
    return suggested_part.replace("_", " ")

# ------ THỬ NGHIỆM ------
prompt = "Bộ Giáo dục và Đào tạo vừa ban hành quy định mới về"
print(f"Prompt: '{prompt}'")

suggested = suggest_next_words(prompt, num_words=12)
print(f"Gợi ý: {suggested}")